# 🎙️ VoiceCloner Simple

**BƯỚC 1:** 3 cells (setup → restart → load model)  
**BƯỚC 2-4:** Upload TXT → Upload giọng → Generate

---
## BƯỚC 1: Setup

In [ ]:
# 1.1 Setup + cài packages
import os, shutil, subprocess
from google.colab import drive

drive.mount('/content/drive')
DRIVE_HF_CACHE = "/content/drive/MyDrive/viterbox_hf_cache"
os.makedirs(DRIVE_HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = DRIVE_HF_CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_HF_CACHE

!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
os.environ['PATH'] = f"/root/.local/bin:{os.environ.get('PATH', '')}"

if os.path.exists('/content/VoiceCloner'):
    shutil.rmtree('/content/VoiceCloner')
!git clone https://github.com/kanazawahere/VoiceCloner.git /content/VoiceCloner --depth=1 --quiet
%cd /content/VoiceCloner

cuda_ver = subprocess.getoutput(r"nvcc --version | grep 'release' | sed 's/.*release //' | sed 's/,.*//' | sed 's/\.//'").strip()
if not cuda_ver.isdigit():
    cuda_ver = "121"

print("⏳ Cài packages (2-3 phút)...")
!uv pip install --system torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu{cuda_ver} --quiet
!uv pip install --system -r requirements.txt --quiet
!uv pip install --system -e . --quiet

print("\n✅ Xong! Chạy cell tiếp để restart.")

In [ ]:
# 1.2 Restart runtime
import os
os.kill(os.getpid(), 9)

In [ ]:
# 1.3 Load model (chạy sau restart)
import os
os.environ['HF_HOME'] = "/content/drive/MyDrive/viterbox_hf_cache"
os.environ['HUGGINGFACE_HUB_CACHE'] = "/content/drive/MyDrive/viterbox_hf_cache"
%cd /content/VoiceCloner

import torch
from viterbox import Viterbox

print("⏳ Load model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
tts = Viterbox.from_pretrained(device)
print(f"✅ Sẵn sàng! ({device})")

---
## BƯỚC 2: Upload TXT

In [ ]:
from google.colab import files

uploaded = files.upload()
txt_file = list(uploaded.keys())[0]

with open(txt_file, 'r', encoding='utf-8') as f:
    text_content = f.read()

print(f"✅ {len(text_content)} ký tự")

---
## BƯỚC 3: Upload giọng mẫu (tùy chọn)

In [ ]:
from google.colab import files

USE_VOICE_CLONING = True  # ← False = giọng mặc định

if USE_VOICE_CLONING:
    uploaded = files.upload()
    ref_audio = list(uploaded.keys())[0]
    print(f"✅ {ref_audio}")
else:
    ref_audio = None
    print("✅ Giọng mặc định")

---
## BƯỚC 4: Generate + tải về

In [ ]:
import librosa, soundfile as sf
from IPython.display import Audio, display
from google.colab import files as colab_files

# ========== CÀI ĐẶT ==========
SPEED = 1.0      # 0.5-1.5
PITCH_SHIFT = 0  # -5 đến +5
# ==============================

print("🎙️ Generate...")
audio = tts.generate(text=text_content, language="vi", audio_prompt=ref_audio, sentence_pause_ms=500)

audio_np = audio.squeeze().cpu().numpy()
sr = 24000

if SPEED != 1.0:
    audio_np = librosa.effects.time_stretch(audio_np, rate=SPEED)
if PITCH_SHIFT != 0:
    audio_np = librosa.effects.pitch_shift(audio_np, sr=sr, n_steps=PITCH_SHIFT)

sf.write("/content/output.wav", audio_np, sr)

print(f"✅ {len(audio_np)/sr:.1f}s")
display(Audio("/content/output.wav"))
colab_files.download("/content/output.wav")